In [ ]:
#!/usr/bin/python3
import sys
import os

sys.path.append(os.path.abspath(".."))

import pickle
import Scripts.utils
from Scripts.utils import *
import numpy as np
from numpy import array
import glob
import pandas as pd
import matplotlib.pyplot as plt
import random, math
import matplotlib.ticker as ticker
import seaborn as sns
from rich.console import Console
from rich.table import Table
from matplotlib.colors import ListedColormap
from sklearn.model_selection import KFold
from tensorflow.keras.utils import plot_model
import visualkeras
from PIL import ImageFont, Image
from sklearn.metrics import r2_score

import copy
import random, math
import time
from joblib import dump
import tensorflow as tf
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from tensorflow.keras import layers, losses
from tensorflow.keras.models import Model
from sklearn.metrics import mean_squared_error as mse
from tensorflow.keras.metrics import RootMeanSquaredError as rmse
from sklearn.model_selection import train_test_split
import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)

import nbformat as nb

np.seterr(divide='ignore', invalid='ignore')

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

dataSamplesGev = np.load('../Data/lzt_dictClus7x7_99k_SignalsSamples_e50_GeV.pkl', allow_pickle=True)
dataAmplitudeGev = np.load('../Data/lzt_dictClus7x7_99k_Amplitudes_e50_GeV.pkl', allow_pickle=True)

# Leitura de elementos do sinal do vetor de amostras
EtruthSampGeV = dataSamplesGev['E'] 
XTcSampGeV = dataSamplesGev['XT_C']
XTlSampGeV = dataSamplesGev['XT_L']
XTrSampGeV = dataSamplesGev['XT_R']
NoiseSampGeV = dataSamplesGev['Noise']

# Sinal de leitura de energia somado as contribuições de Crosstalk e ruído
xDataSamplesGeV = np.add(np.add(np.add(np.add(EtruthSampGeV, XTcSampGeV), XTlSampGeV), NoiseSampGeV), XTrSampGeV)

# Leitura de elementos do sinal do vetor de amplitudes
EtruthAmpGeV = dataAmplitudeGev['E']
XTcAmpGeV = dataAmplitudeGev['XT_C']
XTlAmpGeV = dataAmplitudeGev['XT_L']
XTrAmpGeV = dataAmplitudeGev['XT_R']
NoiseAmpGeV = dataAmplitudeGev['Noise']

# Sinal de leitura de energia somado as contribuições de Crosstalk e ruído
xDataAmpGeV = np.add(np.add(np.add(np.add(EtruthAmpGeV, XTcAmpGeV), XTlAmpGeV), NoiseAmpGeV), XTrAmpGeV)

AmpTimeGeV  = OptFilt(EtruthSampGeV)
AmpTimeXTGeV  = OptFilt(xDataSamplesGeV)

AmplitudesGeVOptFilt = AmpTimeGeV['Clusters']['Amplitude']
XTAmplitudesGeVOptFilt = AmpTimeXTGeV['Clusters']['Amplitude']

shape_val = xDataSamplesGeV.shape[0]

TimesGeVOptFilt = AmpTimeGeV['Clusters']['Time']

scaler_x_data = MinMaxScaler()

xDataSamplesGeV_flat = xDataSamplesGeV.reshape((shape_val, -1))
xDataSamplesGeV_Normalized = scaler_x_data.fit_transform(xDataSamplesGeV_flat)
xDataSamplesGeV_Normalized = xDataSamplesGeV_Normalized.reshape((shape_val,7,7,4))


scaler_data_amp = MinMaxScaler()
scaler_etruth = MinMaxScaler()
scaler_time = MinMaxScaler()
scaler_samp = MinMaxScaler()
scaler_data_samp = MinMaxScaler()
xDataAmpGeV_Normalized = scaler_data_amp.fit_transform(xDataAmpGeV)
EtruthAmpGeV_Normalized = scaler_etruth.fit_transform(EtruthAmpGeV)
TimesGeVOptFilt_Normalized = scaler_time.fit_transform(TimesGeVOptFilt)


xDataSampGeV_Normalized = scaler_data_samp.fit_transform(xDataSamplesGeV)
EtruthSampGeV_Normalized = scaler_samp.fit_transform(EtruthSampGeV)

toGeV = 1000

ij_cell = ['-3,3' , '-2,3' , '-1,3' , '0,3' , '1,3' , '2,3' , '3,3' , 
           '-3,2' , '-2,2' , '-1,2' , '0,2' , '1,2' , '2,2' , '3,2' , 
           '-3,1' , '-2,1' , '-1,1' , '0,1' , '1,1' , '2,1' , '3,1' , 
           '-3,0' , '-2,0' , '-1,0' , '0,0' , '1,0' , '2,0' , '3,0' , 
           '-3,-1', '-2,-1', '-1,-1', '0,-1', '1,-1', '2,-1', '3,-1', 
           '-3,-2', '-2,-2', '-1,-2', '0,-2', '1,-2', '2,-2', '3,-2', 
           '-3,-3', '-2,-3', '-1,-3', '0,-3', '1,-3', '2,-3', '3,-3' ]



Scaled_EtruthSampGeV = EtruthSampGeV.astype('float32')/toGeV
Scaled_xDataSampGeV_Normalized = xDataSampGeV_Normalized.astype('float32')

E_truth_energy_toConv = Scaled_EtruthSampGeV.reshape(shape_val * 4, 7, 7, 1)
XTData_denoise_toConv = xDataSampGeV_Normalized.reshape(shape_val * 4, 7, 7, 1)

group = XTData_denoise_toConv.reshape(shape_val, 4, 7, 7, 1)

stacked = group.transpose(0,2,3,4,1)

XT_Data_To_Conv = stacked.reshape(shape_val, 7, 7, 4)

time_corrupt = AmpTimeXTGeV['Clusters']['Time']


## Divisão dos Dados e Normalização

Inicialmente, os dados de entrada são definidos a partir do conjunto `XT_Data_To_Conv`, enquanto as saídas correspondem aos tempos normalizados armazenados em `TimesGeVOptFilt_Normalized`.

```python
X = XT_Data_To_Conv
y = TimesGeVOptFilt_Normalized
```

Para a avaliação adequada do modelo, o conjunto de dados é dividido em subconjuntos de treinamento, validação e teste.
```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)
```
Em seguida, o conjunto de treinamento é separado para validação
```python
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.1,
    random_state=42
)
```

### Normalização dos Dados

Os dados são normalizados utilizando a técnica Min-Max Scaling.

Os escaladores são definidos como:

```python
scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()
```

O ajuste dos escaladores é realizado exclusivamente com os dados de treinamento.
```python
scaler_x.fit(X_train.reshape(len(X_train), -1))
scaler_y.fit(y_train)
```

No caso das entradas, cada objeto possui dimensão $7 \times 7 \times 4$. Para que a normalização possa ser aplicada, os dados são temporariamente reorganizados em vetores unidimensionais.
```python
def scale_X(X, scaler):
    Xn = scaler.transform(X.reshape(len(X), -1))
    return Xn.reshape(len(X), 7, 7, 4)
```

Para as saídas, a normalização é aplicada diretamente aos vetores de tempos:

```python
def scale_y(y, scaler):
    return scaler.transform(y.reshape(len(y), -1))
```

In [ ]:
X = XT_Data_To_Conv           
y = TimesGeVOptFilt_Normalized  

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.1,
    random_state=42
)

def scale_X(X, scaler):
    Xn = scaler.transform(X.reshape(len(X), -1))
    return Xn.reshape(len(X), 7, 7, 4)

def scale_y(y, scaler):
    return scaler.transform(y.reshape(len(y), -1))

scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()

scaler_x.fit(X_train.reshape(len(X_train), -1))
scaler_y.fit(y_train)

## Otimização de Hiperparâmetros com Optuna

O objetivo desse processo é identificar automaticamente a combinação de parâmetros que proporciona o melhor desempenho do modelo no conjunto de validação.

Inicialmente, são criados os objetos responsáveis pela normalização das entradas e saídas:

```python
scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()

scaler_x.fit(X_train.reshape(len(X_train), -1))
scaler_y.fit(y_train)
```
### Processo de Otimização

A otimização é realizada por meio do método `optimize_with_optuna`, responsável por executar múltiplos experimentos (*trials*)

```python
study = ConvDenoisingAutoencoderV3.optimize_with_optuna(
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    scaler_x=scaler_x,
    scaler_y=scaler_y,
    n_trials=40,
    max_epochs=30
)
```

Em cada *trial*, o Optuna seleciona automaticamente valores para os hiperparâmetros definidos no espaço de busca da arquitetura.

### Salvamento do Estudo

Após a conclusão da busca, o objeto `study` é serializado utilizando a biblioteca `pickle`, permitindo que os resultados sejam recuperados posteriormente sem a necessidade de repetir o processo de otimização.

```python
with open("study_time_v3_with_bi.pkl", "wb") as f:
    pickle.dump(study, f)
```

O arquivo gerado contém todas as informações do estudo, incluindo os hiperparâmetros avaliados, métricas obtidas e a melhor configuração encontrada.

### Exportação dos Resultados

Para facilitar análises posteriores, os resultados de todos os experimentos são convertidos para um `DataFrame` e exportados para um arquivo no formato CSV.

```python
df = study.trials_dataframe()
df.to_csv("optuna_results_time.csv", index=False)
```

O arquivo gerado contém informações detalhadas de cada *trial*, tais como:

- Identificador do experimento;
- Valor da função objetivo;
- Estado do experimento (concluído, podado ou interrompido);
- Hiperparâmetros avaliados;
- Métricas de desempenho associadas.

In [ ]:

scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()

scaler_x.fit(X_train.reshape(len(X_train), -1))
scaler_y.fit(y_train)

study = ConvDenoisingAutoencoderV3.optimize_with_optuna(
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    scaler_x=scaler_x,
    scaler_y=scaler_y,
    n_trials=40,
    max_epochs=30
)

with open("study_time_v3_with_bi.pkl", "wb") as f:
    pickle.dump(study, f)

df = study.trials_dataframe()
df.to_csv("optuna_results_time.csv", index=False)

## Treinamento do Modelo Selecionado

Após a conclusão da etapa de otimização de hiperparâmetros, a configuração que apresentou o melhor desempenho no conjunto de validação é utilizada para instanciar e treinar o modelo final de autoencoder convolucional com remoção de ruído.

A arquitetura é definida por meio da classe `ConvDenoisingAutoencoderV3`, utilizando os hiperparâmetros obtidos durante o processo de otimização:

In [ ]:
dae_build = ConvDenoisingAutoencoderV3(filters=(83, 89),
    kernel_size=(3, 3),
    batch_size=256,
    validation_split=0.1,
    loss_func="mse",
    optimizing_func=tf.keras.optimizers.Adam(learning_rate=0.0016347180445665228),
    activation_func="relu",
    epochs = 30,
    up_sampling=4,
    max_pooling=3,)
history = dae_build.train(X_train, y_train, X_val, y_val)

## Avaliação do Modelo em Escala Física
```python
y_true_real = scaler_time.inverse_transform(
    TimesGeVOptFilt_Normalized.reshape(
        len(TimesGeVOptFilt_Normalized), -1
    )
)
```

```python
y_pred_real = scaler_time.inverse_transform(
    dae_build.model.predict(XT_Data_To_Conv)
    .reshape(len(TimesGeVOptFilt_Normalized), -1)
)
```

### Coeficiente de Determinação ($R^2$)

Para quantificar a qualidade das previsões, é calculado o coeficiente de determinação ($R^2$), uma métrica amplamente utilizada em problemas de regressão.

```python
r2_real = r2_score(y_true_real, y_pred_real)
```

O coeficiente $R^2$ mede a proporção da variabilidade dos dados observados que pode ser explicada pelo modelo. Seu valor é definido por:

:contentReference[oaicite:0]{index=0}

onde:

Valores de $R^2$ próximos de 1 indicam elevada concordância entre os dados reais e as previsões da rede neural, enquanto valores próximos de 0 indicam baixa capacidade explicativa.

### Análise por Mapas de Calor

Além da métrica global de desempenho, é realizada uma análise visual por meio de mapas de calor bidimensionais.

```python
heatmap_all_cells(
    y_true_real,
    y_pred_real,
    bins=120,
    xlim=(-2,2),
    ylim=(-2,2)
)
```

Essa função gera distribuições bidimensionais comparando os valores reais e preditos para cada uma das 49 células do agrupamento $7 \times 7$. Cada mapa de calor representa a densidade de eventos observada na relação entre valor verdadeiro e valor previsto.

In [ ]:
y_true_real = scaler_time.inverse_transform(
    TimesGeVOptFilt_Normalized.reshape(len(TimesGeVOptFilt_Normalized), -1)
)

y_pred_real = scaler_time.inverse_transform(
    dae_build.model.predict(XT_Data_To_Conv).reshape(len(TimesGeVOptFilt_Normalized), -1)
)

r2_real = r2_score(y_true_real, y_pred_real)
heatmap_all_cells(y_true_real, y_pred_real, bins=120, xlim=(-2,2), ylim=(-2,2)) 

## Salva o modelo na extensão .h5

In [ ]:
dae_build.model.save("dae_model_opt.h5")
plot_model(dae_build.model, to_file='model_architecture_opt_filt.png', show_shapes=True, show_layer_names=True)

## Extração de RMSE 

In [ ]:
rmse_cells = []

for i in range(y_true_real.shape[1]):
    rmse_i = np.sqrt(
        mean_squared_error(
            y_true_real[:, i],
            y_pred_real[:, i]
        )
    )
    rmse_cells.append(rmse_i)

rmse_cells = np.array(rmse_cells)
rmse_map = rmse_cells.reshape(7, 7)

rmse_cells = np.array(rmse_cells)  # (49,)

fig, axes = plt.subplots(7, 7, figsize=(10, 10))
axes = axes.flatten()

for i, ax in enumerate(axes):
    ax.set_facecolor("white")

    # escreve o RMSE no centro da célula
    ax.text(
        0.5, 0.5,
        f"{rmse_cells[i]:.3f}",
        ha="center",
        va="center",
        fontsize=10,
        fontweight="bold"
    )

    ax.set_title(f"Célula {i+1}", fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(True)

plt.suptitle("RMSE por célula (matriz 7×7)", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## Gerar Histogramas para a célula escolhida individualmente

In [ ]:

## plotting for hotest cell according to cell addressing
cell = 24

unit = 'GeV'
if unit.lower() == 'gev': toGeV = 1
else: toGeV = 1

#plotHisto(map_EtruthAmp_cells_cropped[:, cell]/toGeV, xDataAmp10GeV_error[:, cell]/toGeV, label='energy', legend=['E', 'E+XT'], unit=unit, text=f'Cell {ij_cell[cell]}')
plotHisto(TimesGeVOptFilt[:, cell], time_corrupt[:, cell], y_pred_real[:, cell], label='time', legend=['Time', 'Time+XT', 'Predicted'], unit=unit, text=f'Cell {ij_cell[cell]}')

## Salva os dados gerados temporais

In [ ]:
import pickle

with open("TimesGeVOptFilt.pkl", "wb") as f:
    pickle.dump(TimesGeVOptFilt, f)

with open("time_corrupt.pkl", "wb") as f:
    pickle.dump(time_corrupt, f)

with open("y_pred_real_com_bi.pkl", "wb") as f:
    pickle.dump(y_pred_real, f)   

## Implementação da validação cruzada

In [ ]:
k_folds_ = 20
shuffle_= True
random_state_ = None
kf = KFold(n_splits=k_folds_, shuffle=shuffle_, random_state=random_state_)
train_losses = []
val_losses = []
all_history = []
fold_results= []
epochs = 30

for train_index, val_index in kf.split(X_train):

    dae_build_cross_val = ConvDenoisingAutoencoderV3(filters=(83, 89),
    kernel_size=(3, 3),
    batch_size=256,
    validation_split=0.1,
    loss_func="mse",
    optimizing_func=tf.keras.optimizers.Adam(learning_rate=0.0016347180445665228),
    activation_func="relu",
    epochs = 30,
    up_sampling=4,
    max_pooling=3,)
    train_data, val_data = X_train[train_index], X_train[val_index]
    train_target, val_target = y_train[train_index], y_train[val_index]
    history = dae_build_cross_val.train(X_train, y_train, X_val, y_val)

    fold_results.append(history)

train_curves = []
val_curves = []

for history_obj in fold_results:
    train_curves.append(history_obj.history["loss"])
    if "val_loss" in history_obj.history:
        val_curves.append(history_obj.history["val_loss"])

min_epochs = min(len(c) for c in train_curves)

train_curves = np.array([c[:min_epochs] for c in train_curves])
val_curves   = np.array([c[:min_epochs] for c in val_curves])

mean_train = np.mean(train_curves, axis=0)
mean_val   = np.mean(val_curves, axis=0)

epochs = range(len(mean_train))

plt.figure(figsize=(12, 6))

plt.plot(epochs, mean_train, label="Average Train Loss")
plt.plot(epochs, mean_val, "--", label="Average Validation Loss")

plt.locator_params(axis='x', nbins=30)
plt.locator_params(axis='y', nbins=10)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend()
plt.grid(True)
plt.show()

## Salva os dados gerados em validação cruzada

In [ ]:
data = {
    "train_loss": mean_train,
    "val_loss": mean_val,
    "epochs": epochs
}

with open("loss_curves.pkl", "wb") as f:
    pickle.dump(data, f)

In [ ]:
train_curves = []
val_curves = []

for history_obj in fold_results:
    train_curves.append(np.array(history_obj.history["loss"]))
    
    if "val_loss" in history_obj.history:
        val_curves.append(np.array(history_obj.history["val_loss"]))
    else:
        val_curves.append(None) 
data = {
    "train_curves": train_curves,
    "val_curves": val_curves
}

with open("loss_curves_full_com_bi.pkl", "wb") as f:
    pickle.dump(data, f)

In [ ]:
# --- Configuração global de estilo ---
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "cm",
    "font.size": 11
})

train_curves = []
val_curves = []

for history_obj in fold_results:
    train_curves.append(history_obj.history["loss"])
    if "val_loss" in history_obj.history:
        val_curves.append(history_obj.history["val_loss"])

min_epochs = min(len(c) for c in train_curves)

train_curves = np.array([c[:min_epochs] for c in train_curves])
val_curves   = np.array([c[:min_epochs] for c in val_curves])

mean_train = np.mean(train_curves, axis=0)
mean_val   = np.mean(val_curves, axis=0)

epochs = np.arange(len(mean_train))

# --- Plot ---
plt.figure(figsize=(10, 5))

plt.plot(epochs, mean_train, linewidth=3.0, label="Loss de treinamento")
plt.plot(epochs, mean_val, linestyle="--", linewidth=3.0, label="Loss de validação")

plt.xlabel("Época")
plt.ylabel("Loss")

plt.minorticks_on()

plt.grid(True, which='major', axis='both',
         linestyle='--', linewidth=0.8, alpha=0.9)

plt.grid(True, which='minor', axis='both',
         linestyle=':', linewidth=0.6, alpha=0.7)

plt.legend(frameon=True, edgecolor="black")

plt.tight_layout()

plt.savefig("loss_curve.pdf", format="pdf", dpi=300)
plt.savefig("loss_curve.png", format="png", dpi=300)

plt.show()